# Diagnostic Distractor v8 — one-shot 8B QLoRA

Run this notebook only after the local preflight in `docs/V8_RUNBOOK.md` reports `training_ready: true`. The frozen v8 benchmark is used only after checkpoint selection; it is never a training or validation source.

Primary target (not guaranteed): GDR ≥90% and ≥40% relative error-rate reduction versus Opus on each pre-registered bounded quality metric. Model-only and verifier-guided best-of-N results are saved separately.

In [ ]:
# STAGE: install
%pip install -q "unsloth==2026.7.1" "datasets>=3.0" "huggingface_hub>=0.30"

import os, sys
from pathlib import Path

if not Path("diagnostic-distractor-slm").exists():
    !git clone https://github.com/jsonjj/diagnostic-distractor-slm.git
%cd diagnostic-distractor-slm
sys.path.insert(0, ".")
print("Repository ready. Confirm this checkout contains data/processed/v8_manifest.json before continuing.")

In [ ]:
# STAGE: configuration
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"
MODEL_REVISION = "1deaf68f694c40dbce295da300851729d759b21a"
TRAIN_FILE = "data/processed/train_v8.jsonl"
FROZEN_BENCHMARK = "data/processed/eval_v8_frozen.jsonl"
LEGACY_DEV = "data/processed/eval_heldout.jsonl"
MANIFEST_FILE = "data/processed/v8_manifest.json"
ADAPTER_DIR = "qwen3-8b-distractor-lora-v8"
HF_REPO_ID = "j2ampn/qwen3-8b-distractor-lora-v8"  # change before Run All if desired
MAX_SEQ_LEN = 2048
EPOCHS = 3
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
BEST_OF_N = 4
SEED = 42

# 8B is the one-shot quality/operability compromise: 4B v7.1 plateaued near 60% binding;
# 14B Q4 plus Unity risks the owner's 16 GB M4 memory/latency budget. Use an L4 or A100.
print({"model": MODEL_NAME, "train": TRAIN_FILE, "epochs": EPOCHS, "best_of_n": BEST_OF_N})

In [ ]:
# STAGE: data-integrity
import hashlib, json
from pathlib import Path
from src.v8_data import assert_no_leakage, validate_training_records, verify_manifest


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


manifest = verify_manifest(MANIFEST_FILE)
if (
    manifest.get("training_ready") is not True
    or manifest.get("deterministic_teacher_filter_ready") is not True
    or manifest.get("opus_judge_ready") is not False
    or manifest.get("opus_teacher_records", 0) < 20
):
    raise RuntimeError(
        "One-shot preflight is incomplete: generate, deterministically filter, and rebuild the Opus teacher pool locally first."
    )
train_records = load_jsonl(TRAIN_FILE)
frozen_records = load_jsonl(FROZEN_BENCHMARK)
legacy_records = load_jsonl(LEGACY_DEV)
assert_no_leakage(train_records, frozen_records)
assert_no_leakage(train_records, legacy_records)
validation_report = validate_training_records(train_records)
if validation_report["pair_consistency"] != 1.0:
    raise RuntimeError("training targets failed hardened verification")

# Group split by full user prompt: repeated/upsampled real questions cannot cross train/validation.
def is_validation(record):
    digest = hashlib.sha256(record["user"].encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % 10 == 0

validation_records = [row for row in train_records if is_validation(row)]
fit_records = [row for row in train_records if not is_validation(row)]
if not validation_records or not fit_records:
    raise RuntimeError("deterministic validation split is empty")
if {row["user"] for row in fit_records} & {row["user"] for row in validation_records}:
    raise RuntimeError("group leakage between fit and validation splits")
print({
    "fit_rows": len(fit_records),
    "validation_rows": len(validation_records),
    "frozen_rows": len(frozen_records),
    "legacy_rows": len(legacy_records),
    "train_sha256": manifest["artifacts"]["train"]["sha256"],
    "frozen_sha256": manifest["artifacts"]["frozen_benchmark"]["sha256"],
})

In [ ]:
# STAGE: base-litmus
import torch
from unsloth import FastLanguageModel
from src.benchmark_v8 import primary_metrics
from src.confidence import ensure_confidence_schema
from src.prompts import SYSTEM_PROMPT, build_user, parse_distractors

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    revision=MODEL_REVISION,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


def sft_to_question(record):
    body = record["user"].split("Question: ", 1)[1]
    body, topic = body.rsplit("\nTopic: ", 1)
    question, correct = body.rsplit("\nCorrect answer: ", 1)
    return {"id": record.get("meta", {}).get("id", hashlib.sha256(record["user"].encode()).hexdigest()[:16]),
            "question": question, "correct": correct, "topic": topic}


def generate_text(current_model, question, *, sample=False, seed=0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user(question["question"], question["correct"], question["topic"])},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt",
    ).to(current_model.device)
    generation_kwargs = {
        "input_ids": inputs,
        "max_new_tokens": 512,
        "do_sample": sample,
    }
    if sample:
        generation_kwargs.update(temperature=0.7, top_p=0.9)
        # Transformers generate() does not universally accept a generator kwarg.
        # Scope and restore global RNG state so each explicit seed stays reproducible.
        cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
        with torch.random.fork_rng(devices=cuda_devices):
            torch.manual_seed(seed)
            if cuda_devices:
                torch.cuda.manual_seed_all(seed)
            output = current_model.generate(**generation_kwargs)
    else:
        output = current_model.generate(**generation_kwargs)
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)


def generate_rows(current_model, questions, *, sample=False, seed_offset=0):
    rows = []
    for index, question in enumerate(questions):
        text = generate_text(current_model, question, sample=sample, seed=seed_offset + index)
        rows.append(ensure_confidence_schema({"id": question["id"], "distractors": parse_distractors(text)}))
    return rows

validation_questions = [sft_to_question(row) for row in validation_records]
base_litmus_questions = validation_questions[:12]
base_litmus = generate_rows(model, base_litmus_questions)
print(json.dumps(primary_metrics(base_litmus_questions, base_litmus), indent=2))

In [ ]:
# STAGE: training
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


def to_text(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)

def training_columns(records):
    return [
        {"system": row["system"], "user": row["user"], "assistant": row["assistant"]}
        for row in records
    ]

fit_dataset = Dataset.from_list(training_columns(fit_records)).map(lambda row: {"text": to_text(row)})
validation_dataset = Dataset.from_list(training_columns(validation_records)).map(lambda row: {"text": to_text(row)})
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=fit_dataset,
    eval_dataset=validation_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=1.5e-4,
        warmup_ratio=0.05,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        seed=SEED,
        output_dir="outputs_v8",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
train_result = trainer.train()  # one and only planned training invocation
print({"best_checkpoint": trainer.state.best_model_checkpoint, "best_eval_loss": trainer.state.best_metric})

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
training_receipt = {
    "schema_version": "diagnostic-distractor-v8-training-v1",
    "base_model": MODEL_NAME,
    "base_revision": MODEL_REVISION,
    "train_sha256": manifest["artifacts"]["train"]["sha256"],
    "frozen_sha256": manifest["artifacts"]["frozen_benchmark"]["sha256"],
    "epochs_planned": EPOCHS,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "lora_r": 32,
    "lora_alpha": 32,
    "learning_rate": 1.5e-4,
    "seed": SEED,
}
Path("v8_training_receipt.json").write_text(
    json.dumps(training_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

# Push in the same Run All only when the owner configured an HF_TOKEN Colab secret.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token and HF_REPO_ID:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    model.push_to_hub(HF_REPO_ID, token=hf_token)
    tokenizer.push_to_hub(HF_REPO_ID, token=hf_token)
    print(f"pushed adapter -> {HF_REPO_ID}")
else:
    print("adapter saved locally; HF push skipped because HF_TOKEN/HF_REPO_ID is unavailable")
FastLanguageModel.for_inference(model)

In [ ]:
# STAGE: model-only

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

model_only_frozen = generate_rows(model, frozen_records)
model_only_legacy = generate_rows(model, legacy_records)
for row in model_only_frozen + model_only_legacy:
    row["generator_model"] = HF_REPO_ID or ADAPTER_DIR
    row["inference_track"] = "model_only"
write_jsonl("predictions_v8_model_only.jsonl", model_only_frozen)
write_jsonl("predictions_v8_model_only_legacy.jsonl", model_only_legacy)

model_only_metrics = primary_metrics(frozen_records, model_only_frozen)
Path("local_metrics_v8_model_only.json").write_text(
    json.dumps(model_only_metrics, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("MODEL-ONLY frozen benchmark local metrics (GDR stays NOT YET RUN until sidecar judging):")
print(json.dumps(model_only_metrics, indent=2))

In [ ]:
# STAGE: best-of-n
from src.consistency import computation_consistent
from src.text_utils import normalize_answer


def verifier_score(question, row):
    distractors = row.get("distractors", [])
    answers = [normalize_answer(item.get("answer", "")) for item in distractors]
    misconceptions = [str(item.get("misconception", "")).strip().casefold() for item in distractors]
    exactly_three = len(distractors) == 3
    key_safe = exactly_three and all(answers) and all(answer != normalize_answer(question["correct"]) for answer in answers)
    distinct_answers = exactly_three and len(set(answers)) == 3 and all(answers)
    distinct_misconceptions = exactly_three and len(set(misconceptions)) == 3 and all(misconceptions)
    computation_passes = sum(
        computation_consistent(
            item.get("computation", ""),
            item.get("answer", ""),
            question["question"],
            display_units=True,
        ) is True
        for item in distractors
    )
    return (int(exactly_three), int(key_safe), int(distinct_answers), int(distinct_misconceptions), computation_passes)


# A rerun restarts only this inference track and cannot leave stale partial results.
for stale_path in (
    Path("predictions_v8_best_of_n.jsonl"),
    Path("local_metrics_v8_best_of_n.json"),
):
    stale_path.unlink(missing_ok=True)

best_of_n_rows = []
for item_index, question in enumerate(frozen_records):
    candidates = [model_only_frozen[item_index]]
    for sample_index in range(1, BEST_OF_N):
        text = generate_text(
            model,
            question,
            sample=True,
            seed=100000 + item_index * BEST_OF_N + sample_index,
        )
        candidates.append(
            ensure_confidence_schema({"id": question["id"], "distractors": parse_distractors(text)})
        )
    chosen = max(candidates, key=lambda row: verifier_score(question, row))
    chosen["generator_model"] = HF_REPO_ID or ADAPTER_DIR
    chosen["inference_track"] = "verifier_guided_best_of_n"
    chosen["best_of_n"] = BEST_OF_N
    best_of_n_rows.append(chosen)

write_jsonl("predictions_v8_best_of_n.jsonl", best_of_n_rows)
best_of_n_metrics = primary_metrics(frozen_records, best_of_n_rows)
Path("local_metrics_v8_best_of_n.json").write_text(
    json.dumps(best_of_n_metrics, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("VERIFIER-GUIDED BEST-OF-N local metrics (reported separately from model-only):")
print(json.dumps(best_of_n_metrics, indent=2))

In [ ]:
# STAGE: download
import shutil

shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
artifacts = [
    f"{ADAPTER_DIR}.zip",
    "predictions_v8_model_only.jsonl",
    "predictions_v8_model_only_legacy.jsonl",
    "predictions_v8_best_of_n.jsonl",
    "local_metrics_v8_model_only.json",
    "local_metrics_v8_best_of_n.json",
    "v8_training_receipt.json",
]
try:
    from google.colab import files
    for artifact in artifacts:
        files.download(artifact)
except ImportError:
    print("Not in Colab; artifacts remain in the working directory.")

print(r'''
Training is complete. Do not retrain to chase metrics.

Run only the LOCAL Opus frontier-generation baseline below. Opus is not the judge.
Always run --estimate-only first and confirm the account-specific price in TrueFoundry:

python -m src.run_frontier --input data/processed/eval_v8_frozen.jsonl --model anthropic-primary/claude-opus-4-8 --out predictions_opus_v8.jsonl --estimate-only
python -m src.run_frontier --input data/processed/eval_v8_frozen.jsonl --model anthropic-primary/claude-opus-4-8 --out predictions_opus_v8.jsonl --max-calls 140

No accepted independent v8 judge artifact currently exists. Report deterministic schema, key-safety, distinctness, and hardened computation/grounding metrics. Leave GDR, task-quality/plausibility proxy, ECE/Brier, and mixed/nonnumeric confidence unavailable. If an independent judge later clears calibration, estimate it separately and apply the identical model, prompt, thresholds, and cache policy to both v8 and Opus; label those quality results as proxies.

Stopping rule: report the measured result. A second training run is contingency-only for a demonstrated technical failure (corrupt data, failed save, wrong checkpoint), not routine iteration.
''')